# Tugas 3 — Data Acquisition

## Analisis Pertumbuhan Ekonomi dan Tingkat Pengangguran Negara G20 Periode 2010–2024

**Nama:** Ayang Nova Anggraeni  
**NIM:** 2310631170126  
**Mata Kuliah:** Data Engineering  
**Materi:** REST API Data Acquisition

## 1. Deskripsi Kasus

Data pertumbuhan ekonomi dan tingkat pengangguran merupakan indikator yang
dapat digunakan untuk menggambarkan kondisi ekonomi suatu negara.

Pada tugas ini dilakukan akuisisi data World Bank untuk 19 negara anggota G20
dengan periode 2010–2024. Data yang dikumpulkan terdiri dari indikator
pertumbuhan GDP tahunan dan tingkat pengangguran.

Akuisisi dilakukan menggunakan World Bank Open Data REST API dengan metode
HTTP GET. Data dari API kemudian diubah dari format JSON menjadi Pandas
DataFrame, dilakukan transformasi dasar dan pemeriksaan kualitas data,
kemudian disimpan dalam format CSV dan Parquet.

## 2. Tujuan

Tujuan dari proses data acquisition ini adalah:

1. Mengambil data ekonomi dari World Bank Open Data menggunakan REST API.
2. Mengambil data untuk beberapa negara dan beberapa indikator.
3. Menerapkan pagination pada API untuk memperoleh seluruh data.
4. Mengubah response JSON menjadi Pandas DataFrame.
5. Melakukan transformasi dan cleaning dasar.
6. Melakukan pemeriksaan kualitas data.
7. Menyimpan dataset akhir dalam format CSV dan Parquet.

## 3. Sumber Data

**Sumber:** World Bank Open Data

Website:
https://data.worldbank.org/

API:
https://api.worldbank.org/v2/

Data diperoleh melalui World Bank REST API v2 menggunakan HTTP GET.

API tidak membutuhkan API key untuk endpoint yang digunakan pada tugas ini.

## 4. Indikator yang Digunakan

| Variabel | Indikator | Kode World Bank | Satuan |
|---|---|---|---|
| GDP Growth | GDP growth (annual %) | `NY.GDP.MKTP.KD.ZG` | Persen (%) |
| Unemployment | Unemployment, total (% of total labor force) | `SL.UEM.TOTL.ZS` | Persen (%) |

### GDP Growth

Kode indikator:

`NY.GDP.MKTP.KD.ZG`

Indikator ini menunjukkan pertumbuhan GDP tahunan dalam persen.

### Unemployment

Kode indikator:

`SL.UEM.TOTL.ZS`

Indikator ini menunjukkan tingkat pengangguran sebagai persentase dari total
angkatan kerja.

## 5. Negara dan Periode

### Negara

Digunakan 19 negara anggota G20, dengan tidak memasukkan EU sebagai agregat:

- Argentina (`ARG`)
- Australia (`AUS`)
- Brazil (`BRA`)
- Canada (`CAN`)
- China (`CHN`)
- France (`FRA`)
- Germany (`DEU`)
- India (`IND`)
- Indonesia (`IDN`)
- Italy (`ITA`)
- Japan (`JPN`)
- Mexico (`MEX`)
- Russia (`RUS`)
- Saudi Arabia (`SAU`)
- South Africa (`ZAF`)
- South Korea (`KOR`)
- Türkiye (`TUR`)
- United Kingdom (`GBR`)
- United States (`USA`)

### Periode

2010–2024

Total observasi yang diharapkan:

`19 negara × 15 tahun × 2 indikator = 570 observasi`

## **Import & Configuration**

In [10]:
from pathlib import Path
import sys

import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    COUNTRIES,
    INDICATORS,
    START_YEAR,
    END_YEAR
)

from src.extract import acquire_worldbank_data
from src.transform import transform_data
from src.validate import validate_data
from src.load import save_data

In [11]:
print("Project root:", PROJECT_ROOT)
print("Countries:", len(COUNTRIES))
print("Indicators:", INDICATORS)
print("Period:", f"{START_YEAR}-{END_YEAR}")

Project root: c:\Semester 7 (2026)\Data Engineering\Tugas 3_DAQ_Ayang Nova Anggraeni_2310631170126
Countries: 19
Indicators: {'gdp_growth': 'NY.GDP.MKTP.KD.ZG', 'unemployment': 'SL.UEM.TOTL.ZS'}
Period: 2010-2024


## **Configuration Overview**

In [12]:
print("Countries :", len(COUNTRIES))
print("Indicators:", INDICATORS)
print("Period    :", f"{START_YEAR}-{END_YEAR}")

Countries : 19
Indicators: {'gdp_growth': 'NY.GDP.MKTP.KD.ZG', 'unemployment': 'SL.UEM.TOTL.ZS'}
Period    : 2010-2024


## 6. Extract — Akuisisi Data dari World Bank API

Data diperoleh menggunakan HTTP GET melalui World Bank REST API.

Parameter yang digunakan meliputi:

- `date`: membatasi periode 2010–2024
- `format`: JSON
- `per_page`: jumlah data per halaman
- `page`: nomor halaman

Program juga menggunakan timeout dan HTTP status checking serta menangani
error dasar pada request dan response JSON.

Pagination dilakukan secara otomatis hingga seluruh halaman API berhasil
diambil.

In [24]:
print("Starting data extraction...\n")

raw_data = acquire_worldbank_data()

print("\nRaw data acquired.")
print(f"Rows: {len(raw_data)}")

Starting data extraction...

Indicator NY.GDP.MKTP.KD.ZG: page 1/3
Indicator NY.GDP.MKTP.KD.ZG: page 2/3
Indicator NY.GDP.MKTP.KD.ZG: page 3/3
Indicator SL.UEM.TOTL.ZS: page 1/3
Indicator SL.UEM.TOTL.ZS: page 2/3
Indicator SL.UEM.TOTL.ZS: page 3/3

Raw data acquired.
Rows: 570


## **Raw Data Preview**

In [14]:
raw_data.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal,indicator_variable
0,"{'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP grow...","{'id': 'AR', 'value': 'Argentina'}",ARG,2024,-1.342931,,,1,gdp_growth
1,"{'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP grow...","{'id': 'AR', 'value': 'Argentina'}",ARG,2023,-1.855788,,,1,gdp_growth
2,"{'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP grow...","{'id': 'AR', 'value': 'Argentina'}",ARG,2022,6.020745,,,1,gdp_growth
3,"{'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP grow...","{'id': 'AR', 'value': 'Argentina'}",ARG,2021,10.441812,,,1,gdp_growth
4,"{'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP grow...","{'id': 'AR', 'value': 'Argentina'}",ARG,2020,-9.900485,,,1,gdp_growth


In [15]:
raw_data.shape

(570, 9)

## 7. Transform — Cleaning dan Transformasi Data

Response dari World Bank memiliki beberapa field dalam bentuk nested object,
seperti `country` dan `indicator`.

Transformasi dilakukan untuk:

1. Mengambil nama negara dari nested object.
2. Mengambil nama indikator dari nested object.
3. Mengubah nama kolom menjadi lebih sederhana.
4. Mengubah `year` menjadi tipe integer nullable.
5. Mengubah `value` menjadi numeric.
6. Mengurutkan data berdasarkan negara, tahun, dan indikator.

In [16]:
final_data = transform_data(raw_data)

final_data.head()

,country_code,country,year,indicator_name,indicator_variable,value
0,ARG,Argentina,2010,GDP growth (annual %),gdp_growth,10.125398
1,ARG,Argentina,2010,"Unemployment, total (% of total labor force) (...",unemployment,7.714000
2,ARG,Argentina,2011,GDP growth (annual %),gdp_growth,6.003952
3,ARG,Argentina,2011,"Unemployment, total (% of total labor force) (...",unemployment,7.180000
4,ARG,Argentina,2012,GDP growth (annual %),gdp_growth,-1.026420


## **Check Struktur Dataset**

In [17]:
print("Shape:", final_data.shape)

print("\nColumns:")
print(final_data.columns.tolist())

print("\nData types:")
print(final_data.dtypes)

Shape: (570, 6)

Columns:
['country_code', 'country', 'year', 'indicator_name', 'indicator_variable', 'value']

Data types:
country_code              str
country                   str
year                    Int64
indicator_name            str
indicator_variable        str
value                 float64
dtype: object


## 8. Data Quality Validation

Pemeriksaan kualitas data dilakukan untuk memastikan dataset hasil akuisisi
memiliki struktur dan isi yang sesuai.

Pemeriksaan meliputi:

- tipe data setiap kolom,
- keberadaan dictionary pada data hasil transformasi,
- jumlah baris dan kolom,
- missing values,
- duplicate observations,
- jumlah negara,
- jumlah observasi setiap indikator,
- rentang tahun.

In [18]:
validate_data(final_data)


=== DATA QUALITY CHECK ===

Column data types:
country_code              str
country                   str
year                    Int64
indicator_name            str
indicator_variable        str
value                 float64
dtype: object

Columns containing dictionaries:
No dictionaries found.

Rows       : 570
Columns    : 6

Missing values:
country_code          0
country               0
year                  0
indicator_name        0
indicator_variable    0
value                 0
dtype: int64

Duplicate observations:
0

Countries:
19

Indicators:
indicator_variable
gdp_growth      285
unemployment    285
Name: count, dtype: int64

Year range:
2010 - 2024


## **Additional Expected Row Check**

In [19]:
expected_rows = (
    len(COUNTRIES)
    * (END_YEAR - START_YEAR + 1)
    * len(INDICATORS)
)

actual_rows = len(final_data)

print(f"Expected rows: {expected_rows}")
print(f"Actual rows  : {actual_rows}")

assert actual_rows == expected_rows

print("Row count validation: PASSED")

Expected rows: 570
Actual rows  : 570
Row count validation: PASSED


## **Verify Output**

In [20]:
csv_path = PROJECT_ROOT / "data" / "hasil_data.csv"
parquet_path = PROJECT_ROOT / "data" / "hasil_data.parquet"

print("CSV exists     :", csv_path.exists())
print("Parquet exists :", parquet_path.exists())

CSV exists     : True
Parquet exists : True


In [21]:
csv_data = pd.read_csv(csv_path)

print("CSV shape:", csv_data.shape)
csv_data.head()

CSV shape: (570, 6)


,country_code,country,year,indicator_name,indicator_variable,value
0,ARG,Argentina,2010,GDP growth (annual %),gdp_growth,10.125398
1,ARG,Argentina,2010,"Unemployment, total (% of total labor force) (...",unemployment,7.714000
2,ARG,Argentina,2011,GDP growth (annual %),gdp_growth,6.003952
3,ARG,Argentina,2011,"Unemployment, total (% of total labor force) (...",unemployment,7.180000
4,ARG,Argentina,2012,GDP growth (annual %),gdp_growth,-1.026420


In [22]:
parquet_data = pd.read_parquet(parquet_path)

print("Parquet shape:", parquet_data.shape)
parquet_data.head()

Parquet shape: (570, 6)


,country_code,country,year,indicator_name,indicator_variable,value
0,ARG,Argentina,2010,GDP growth (annual %),gdp_growth,10.125398
1,ARG,Argentina,2010,"Unemployment, total (% of total labor force) (...",unemployment,7.714000
2,ARG,Argentina,2011,GDP growth (annual %),gdp_growth,6.003952
3,ARG,Argentina,2011,"Unemployment, total (% of total labor force) (...",unemployment,7.180000
4,ARG,Argentina,2012,GDP growth (annual %),gdp_growth,-1.026420


## **Acquisition Timestamp**

In [23]:
from datetime import datetime

acquisition_timestamp = datetime.now().astimezone()

print(
    "Acquisition timestamp:",
    acquisition_timestamp.isoformat()
)

Acquisition timestamp: 2026-09-12T00:24:27.091238+07:00


## 9. Keterbatasan Data

Beberapa keterbatasan dalam proses akuisisi ini adalah:

1. Data bergantung pada ketersediaan dan pembaruan data dari World Bank.
2. Beberapa observasi dapat berubah apabila World Bank melakukan revisi atau
   pembaruan terhadap data historis.
3. Dataset hanya mencakup 19 negara G20 dan periode 2010–2024.
4. Data yang digunakan terbatas pada dua indikator, yaitu GDP growth dan
   unemployment.
5. Tugas ini berfokus pada proses data acquisition, sehingga tidak dilakukan
   analisis statistik atau pemodelan lebih lanjut.

## 10. Kesimpulan

Proses data acquisition berhasil dilakukan menggunakan World Bank Open Data
REST API.

Dataset akhir terdiri dari:

- 19 negara,
- 2 indikator,
- periode 2010–2024,
- total 570 observasi.

Data berhasil diperoleh melalui REST API dengan HTTP GET, pagination,
timeout handling, HTTP status checking, dan basic error handling.

Response JSON berhasil dikonversi menjadi Pandas DataFrame, kemudian dilakukan
transformasi dan validasi data. Hasil akhir berhasil disimpan dalam format
CSV dan Parquet.

Dengan demikian, seluruh tahapan data acquisition dari extraction hingga
penyimpanan dataset berhasil dilakukan.